In [4]:
import numpy as np

# Number of states
n = int(input("Enter number of hidden states: "))

# State names
states = input(f"Enter {n} state names separated by space: ").split()
while len(states) != n:
    print(f"Please enter exactly {n} state names.")
    states = input(f"Enter {n} state names separated by space: ").split()

# Number of observation symbolsewwe
m = int(input("\nEnter number of observation symbols: "))

# Observation symbols
obs_symbols = input(f"Enter {m} observation symbols separated by space: ").split()
while len(obs_symbols) != m:
    print(f"Please enter exactly {m} observation symbols.")
    obs_symbols = input(f"Enter {m} observation symbols separated by space: ").split()

# Initial probabilities
pi = list(map(float, input(f"\nEnter {n} initial probabilities: ").split()))
while len(pi) != n:
    print(f"Please enter exactly {n} values.")
    pi = list(map(float, input(f"Enter {n} initial probabilities: ").split()))
pi = np.array(pi)

# Transition Matrix
print("\nEnter Transition Probability Matrix")
A = []

for i in range(n):
    row = list(map(float, input(f"Transition from {states[i]} to {' '.join(states)}: ").split()))
    while len(row) != n:
        print(f"Please enter exactly {n} values.")
        row = list(map(float, input(f"Transition from {states[i]} to {' '.join(states)}: ").split()))
    A.append(row)

A = np.array(A)

# Emission Matrix
print("\nEnter Emission Probability Matrix")
B = []

for i in range(n):
    row = list(map(float, input(f"Emission probabilities for {states[i]} ({' '.join(obs_symbols)}): ").split()))
    while len(row) != m:
        print(f"Please enter exactly {m} values.")
        row = list(map(float, input(f"Emission probabilities for {states[i]} ({' '.join(obs_symbols)}): ").split()))
    B.append(row)

B = np.array(B)

# Observation sequence
T = int(input("\nEnter length of observation sequence: "))

obs_sequence = input("Enter observation sequence: ").split()
while len(obs_sequence) != T:
    print(f"Please enter exactly {T} observations.")
    obs_sequence = input("Enter observation sequence: ").split()

# Convert observations to indices
obs_index = {obs: i for i, obs in enumerate(obs_symbols)}

try:
    O = [obs_index[x] for x in obs_sequence]
except KeyError as e:
    print(f"Invalid observation: {e}")
    exit()

# Forward Algorithm
alpha = np.zeros((T, n))

# Initialization
for i in range(n):
    alpha[0][i] = pi[i] * B[i][O[0]]

# Recursion
for t in range(1, T):
    for j in range(n):
        alpha[t][j] = np.sum(alpha[t-1] * A[:, j]) * B[j][O[t]]

# Termination
likelihood = np.sum(alpha[T-1])

# Display Forward Table
print("\nForward Probability Table\n")

for t in range(T):
    print(f"Time {t+1} ({obs_sequence[t]})")
    for i in range(n):
        print(f"{states[i]} : {alpha[t][i]:.8f}")
    print()

print("Likelihood of Observation Sequence =", likelihood)

Enter number of hidden states:  3
Enter 3 state names separated by space:  walk run rest

Enter number of observation symbols:  3
Enter 3 observation symbols separated by space:  low med high

Enter 3 initial probabilities:  0.5 0.3 0.2



Enter Transition Probability Matrix


Transition from walk to walk run rest:  0.6 0.3 0.1
Transition from run to walk run rest:  0.2 0.5 0.3
Transition from rest to walk run rest:  0.3 0.2 0.5



Enter Emission Probability Matrix


Emission probabilities for walk (low med high):  0.6 0.3 0.1
Emission probabilities for run (low med high):  0.1 0.3 0.6
Emission probabilities for rest (low med high):  0.7 0.2 0.1

Enter length of observation sequence:  4
Enter observation sequence:  high med low high



Forward Probability Table

Time 1 (high)
walk : 0.05000000
run : 0.18000000
rest : 0.02000000

Time 2 (med)
walk : 0.02160000
run : 0.03270000
rest : 0.01380000

Time 3 (low)
walk : 0.01418400
run : 0.00255900
rest : 0.01320900

Time 4 (high)
walk : 0.00129849
run : 0.00490590
rest : 0.00087906

Likelihood of Observation Sequence = 0.007083450000000001


In [ ]:
import numpy as np
from hmmlearn import hmm

# Number of hidden states
n = int(input("Enter number of hidden states: "))

# State names
states = input(f"Enter {n} state names separated by space: ").split()

# Number of observation symbols
m = int(input("Enter number of observation symbols: "))

# Observation symbols
obs_symbols = input(f"Enter {m} observation symbols separated by space: ").split()

# Initial probabilities
start_prob = np.array(list(map(float, input(f"Enter {n} initial probabilities: ").split())))

# Transition matrix
print("\nEnter Transition Probability Matrix")
trans_mat = []

for i in range(n):
    row = list(map(float, input(f"Transition from {states[i]} to {' '.join(states)}: ").split()))
    trans_mat.append(row)

trans_mat = np.array(trans_mat)

# Emission matrix
print("\nEnter Emission Probability Matrix")
emission_mat = []

for i in range(n):
    row = list(map(float, input(f"Emission probabilities for {states[i]} ({' '.join(obs_symbols)}): ").split()))
    emission_mat.append(row)

emission_mat = np.array(emission_mat)

# Observation mapping
obs_map = {obs: i for i, obs in enumerate(obs_symbols)}

# Create HMM
model = hmm.CategoricalHMM(n_components=n, init_params="")

model.startprob_ = start_prob
model.transmat_ = trans_mat
model.emissionprob_ = emission_mat

# Observation sequence
T = int(input("\nEnter length of observation sequence: "))
obs_seq = input("Enter observation sequence: ").split()

X = np.array([obs_map[x] for x in obs_seq]).reshape(-1,1)

# Likelihood
log_prob = model.score(X)

print("\nLog Likelihood =", log_prob)
print("Likelihood =", np.exp(log_prob))

# Most probable hidden states
hidden = model.predict(X)

print("\nMost Probable Hidden State Sequence")

for o,s in zip(obs_seq, hidden):
    print(o,"->",states[s])

# Train using Baum-Welch
print("\nTraining using Baum-Welch...")

trained_model = hmm.CategoricalHMM(
    n_components=n,
    n_iter=100,
    random_state=42
)

trained_model.fit(X)

print("\nLearned Initial State Probabilities")
print(trained_model.startprob_)

print("\nLearned Transition Matrix")
print(trained_model.transmat_)

print("\nLearned Emission Matrix")
print(trained_model.emissionprob_)

# Additional observation sequences
k = int(input("\nHow many additional observation sequences? "))

for i in range(k):
    print(f"\nSequence {i+1}")
    l = int(input("Length: "))
    seq = input("Enter observations: ").split()

    X_new = np.array([obs_map[x] for x in seq]).reshape(-1,1)

    print("Likelihood =", np.exp(model.score(X_new)))